In [1]:
import pm4py

log = pm4py.read_xes("BPI_Challenge_2012.xes.gz")
df = pm4py.convert_to_dataframe(log)

print(df.shape)
print(df.columns.tolist())
print(df["case:concept:name"].nunique(), "cases")
print(df["concept:name"].value_counts().head(15))



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing


c:\Users\dohuu\miniconda3\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\dohuu\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 13087/13087 [00:11<00:00, 1179.82it/s]


(262200, 7)
['org:resource', 'lifecycle:transition', 'concept:name', 'time:timestamp', 'case:REG_DATE', 'case:concept:name', 'case:AMOUNT_REQ']
13087 cases
concept:name
W_Completeren aanvraag            54850
W_Nabellen offertes               52016
W_Nabellen incomplete dossiers    25190
W_Valideren aanvraag              20809
W_Afhandelen leads                16566
A_SUBMITTED                       13087
A_PARTLYSUBMITTED                 13087
A_DECLINED                         7635
A_PREACCEPTED                      7367
O_SELECTED                         7030
O_CREATED                          7030
O_SENT                             7030
A_ACCEPTED                         5113
A_FINALIZED                        5015
O_CANCELLED                        3655
Name: count, dtype: int64


In [2]:
import pandas as pd
import pm4py

df = df.sort_values(["case:concept:name", "time:timestamp"])

# 1. Mỗi hoạt động có những trạng thái lifecycle nào
print(df.groupby(["concept:name", "lifecycle:transition"]).size().unstack(fill_value=0))

# 2. Thời gian xử lý mỗi hồ sơ (ngày)
case = df.groupby("case:concept:name")["time:timestamp"].agg(["min", "max"])
case["days"] = (case["max"] - case["min"]).dt.total_seconds() / 86400
print(case["days"].describe())

# 3. Kết quả cuối của từng hồ sơ
final = (df[df["concept:name"].isin(["A_DECLINED", "A_CANCELLED", "A_APPROVED"])]
         .groupby("case:concept:name")["concept:name"].last())
print(final.value_counts())

# 4. Rework: hồ sơ phải "hoàn thiện" nhiều hơn 1 lần
w = df[(df["concept:name"] == "W_Completeren aanvraag")
       & (df["lifecycle:transition"] == "COMPLETE")]
n = w.groupby("case:concept:name").size()
print("Ty le ho so lam lai >1 lan:", (n > 1).mean())

# 5. Số variant (số đường đi khác nhau của quy trình)
print("So variant:", len(pm4py.get_variants(log)))

lifecycle:transition            COMPLETE  SCHEDULE  START
concept:name                                             
A_ACCEPTED                          5113         0      0
A_ACTIVATED                         2246         0      0
A_APPROVED                          2246         0      0
A_CANCELLED                         2807         0      0
A_DECLINED                          7635         0      0
A_FINALIZED                         5015         0      0
A_PARTLYSUBMITTED                  13087         0      0
A_PREACCEPTED                       7367         0      0
A_REGISTERED                        2246         0      0
A_SUBMITTED                        13087         0      0
O_ACCEPTED                          2243         0      0
O_CANCELLED                         3655         0      0
O_CREATED                           7030         0      0
O_DECLINED                           802         0      0
O_SELECTED                          7030         0      0
O_SENT        

In [3]:
# 1. Thời gian xử lý theo kết quả cuối
case["outcome"] = final.reindex(case.index).fillna("NO_FINAL")
print(case.groupby("outcome")["days"].describe())

# 2. Top 10 variant phủ bao nhiêu % hồ sơ
seq = df.groupby("case:concept:name")["concept:name"].apply(tuple)
vc = seq.value_counts()
print("Top 10 variants phu:", vc.head(10).sum() / vc.sum())

# 3. Thời gian làm thực tế của từng công việc W_ (START -> COMPLETE, phút)
w = df[df["concept:name"].str.startswith("W_")
       & df["lifecycle:transition"].isin(["START", "COMPLETE"])].copy()
w["k"] = w.groupby(["case:concept:name", "concept:name", "lifecycle:transition"]).cumcount()
p = (w.set_index(["case:concept:name", "concept:name", "k", "lifecycle:transition"])
       ["time:timestamp"].unstack())
p["minutes"] = (p["COMPLETE"] - p["START"]).dt.total_seconds() / 60
print(p.groupby("concept:name")["minutes"].describe())

# 4. Số người/hệ thống xử lý
print(df["org:resource"].nunique())
print(df["org:resource"].value_counts().head(5))

              count       mean        std       min        25%        50%  \
outcome                                                                     
A_APPROVED   2246.0  16.735392   9.778055  0.008199  10.637065  14.485519   
A_CANCELLED  2807.0  18.599560  14.562737  0.000771   6.747935  16.846095   
A_DECLINED   7635.0   2.048094   5.995068  0.000021   0.000447   0.007829   
NO_FINAL      399.0  18.612507  10.522635  0.491942  11.814363  16.074627   

                   75%         max  
outcome                             
A_APPROVED   19.987844  137.221482  
A_CANCELLED  30.770678   91.455280  
A_DECLINED    0.491189   76.946342  
NO_FINAL     22.505567   67.264538  
Top 10 variants phu: 0.493237563994804
                                  count       mean         std          min  \
concept:name                                                                  
W_Afhandelen leads               5897.0  16.943925  163.924465     0.011217   
W_Beoordelen fraude               270.0

In [4]:
import pandas as pd

# 1. Làm sạch thời gian làm việc và ước lượng khối lượng
p = p.reset_index()
ok = p[p["minutes"] > 0].copy()
print("Ty le cap bi loai:", 1 - len(ok) / p["minutes"].notna().sum())
cap = ok.groupby("concept:name")["minutes"].transform(lambda s: s.quantile(0.99))
ok["m"] = ok["minutes"].clip(upper=cap)
tot = ok.groupby("concept:name")["m"].sum() / 60
print(tot.round(0))
print((tot / tot.sum()).round(3))

# 2. Công sức dồn vào hồ sơ có kết quả nào
ok["outcome"] = ok["case:concept:name"].map(final).fillna("NO_FINAL")
print(ok.groupby("outcome")["m"].sum().div(60).round(0))

# 3. Resource 112 làm gì
print(df[df["org:resource"].astype(str) == "112"]["concept:name"].value_counts().head(10))

# 4. Top 5 variant (gộp các bước lặp liên tiếp) để vẽ AS-IS
def collapse(s):
    out = []
    for a in s:
        if not out or out[-1] != a:
            out.append(a)
    return " → ".join(out)

seq_c = (df[df["lifecycle:transition"] == "COMPLETE"]
         .groupby("case:concept:name")["concept:name"].apply(collapse))
for s, n in seq_c.value_counts().head(5).items():
    print(n, "|", s)

Ty le cap bi loai: 5.6043601922350916e-05
concept:name
W_Afhandelen leads                 660.0
W_Beoordelen fraude                  5.0
W_Completeren aanvraag            2223.0
W_Nabellen incomplete dossiers    1128.0
W_Nabellen offertes               1116.0
W_Valideren aanvraag              2172.0
Name: m, dtype: float64
concept:name
W_Afhandelen leads                0.090
W_Beoordelen fraude               0.001
W_Completeren aanvraag            0.304
W_Nabellen incomplete dossiers    0.154
W_Nabellen offertes               0.153
W_Valideren aanvraag              0.297
Name: m, dtype: float64
outcome
A_APPROVED     3389.0
A_CANCELLED    1801.0
A_DECLINED     1775.0
NO_FINAL        340.0
Name: m, dtype: float64
concept:name
A_SUBMITTED               13087
A_PARTLYSUBMITTED         13087
W_Completeren aanvraag     4853
A_PREACCEPTED              4852
W_Afhandelen leads         4739
A_DECLINED                 3429
A_CANCELLED                1004
O_CANCELLED                 560
W_Beoorde

In [5]:
# 1. Variant phổ biến của hồ sơ được duyệt (để vẽ AS-IS luồng thành công)
appr = case.index[case["outcome"] == "A_APPROVED"]
seq_a = seq_c.loc[seq_c.index.isin(appr)]
for s, n in seq_a.value_counts().head(5).items():
    print(n, "|", s)

# 2. Công việc W_ hoàn tất SAU quyết định cuối
dec = (df[df["concept:name"].isin(["A_DECLINED", "A_CANCELLED", "A_APPROVED"])]
       .groupby("case:concept:name")["time:timestamp"].max().rename("t_dec"))
wc = df[df["concept:name"].str.startswith("W_") & (df["lifecycle:transition"] == "COMPLETE")]
wc = wc.join(dec, on="case:concept:name")
after = wc[wc["time:timestamp"] > wc["t_dec"]]
print("Ho so co viec W_ xong sau quyet dinh:", after["case:concept:name"].nunique())
print(after["concept:name"].value_counts())

# 3. Thời gian chờ giữa 2 sự kiện liên tiếp: chỗ nào chờ lâu nhất
d = (df[df["lifecycle:transition"] == "COMPLETE"]
     .sort_values(["case:concept:name", "time:timestamp"]).copy())
d["next"] = d.groupby("case:concept:name")["concept:name"].shift(-1)
d["wait_h"] = (d.groupby("case:concept:name")["time:timestamp"].shift(-1)
               - d["time:timestamp"]).dt.total_seconds() / 3600
w = d.groupby(["concept:name", "next"])["wait_h"].agg(["count", "median"])
print(w[w["count"] >= 500].sort_values("median", ascending=False).head(10))

14 | A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → W_Completeren aanvraag → W_Nabellen offertes → O_SENT_BACK → W_Nabellen offertes → O_ACCEPTED → A_REGISTERED → A_APPROVED → A_ACTIVATED → W_Valideren aanvraag
13 | A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → A_FINALIZED → O_SELECTED → O_CREATED → O_SENT → W_Completeren aanvraag → W_Nabellen offertes → O_SENT_BACK → W_Nabellen offertes → O_ACCEPTED → A_REGISTERED → A_APPROVED → A_ACTIVATED → W_Valideren aanvraag
12 | A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → W_Completeren aanvraag → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → W_Completeren aanvraag → W_Nabellen offertes → O_SENT_BACK → W_Nabellen offertes → O_ACCEPTED → A_APPROVED → A_REGISTERED → A_ACTIVATED → W_Valideren aanvraag
11 | A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → W_Completeren aanvraag → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → W_Complet

In [6]:
# Công việc BẮT ĐẦU sau quyết định (mới là việc làm sau khi đã quyết định)
st = (df[df["concept:name"].str.startswith("W_") & (df["lifecycle:transition"] == "START")]
      .join(dec, on="case:concept:name"))
st = st[st["time:timestamp"] > st["t_dec"]]
print("Viec BAT DAU sau quyet dinh:", len(st), "| ho so:", st["case:concept:name"].nunique())
print(st["concept:name"].value_counts())
print(st["case:concept:name"].map(final).fillna("NO_FINAL").value_counts())

# Độ trễ giữa quyết định và lúc đóng công việc (giờ)
after = after.copy()
after["lag_h"] = (after["time:timestamp"] - after["t_dec"]).dt.total_seconds() / 3600
print(after.groupby("concept:name")["lag_h"].describe()[["count", "25%", "50%", "75%"]])

Viec BAT DAU sau quyet dinh: 72 | ho so: 7
concept:name
W_Nabellen offertes               40
W_Nabellen incomplete dossiers    23
W_Valideren aanvraag               8
W_Completeren aanvraag             1
Name: count, dtype: int64
case:concept:name
A_APPROVED     49
A_CANCELLED    23
Name: count, dtype: int64
                                 count       25%       50%       75%
concept:name                                                        
W_Afhandelen leads              2234.0  0.000857  0.000948  0.001087
W_Beoordelen fraude               57.0  0.000629  0.000717  0.000854
W_Completeren aanvraag          1852.0  0.000525  0.000867  0.001021
W_Nabellen incomplete dossiers   431.0  0.000939  0.001694  0.002333
W_Nabellen offertes             1055.0  0.000595  0.000691  0.000862
W_Valideren aanvraag            2753.0  0.001565  0.002021  0.002276
